In [1]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_PRIVATE_CREDIT")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [8]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_private_creit_links.parquet", index=False)

In [9]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 171


In [10]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\((\[\[.*?\]\])\)'

sleep_duration = 5

In [12]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [13]:
start_time = time.time()
for url in country_links:
    # Extraer país desde el URL
    country = urlparse(url).path.strip("/").split("/")[0]

    # Verificar si el país está en la lista válida
    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            df_pivot = df.pivot(index="Period", columns="Country", values="Value")
            master_df = pd.merge(master_df, df_pivot, how="outer", left_index=True, right_index=True)

            print(f"✅ Datos extraídos correctamente para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Angola no está en la lista, se omite.
⏭️ País Antigua-and-Barbuda no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
⏭️ País Aruba no está en la lista, se omite.
✅ Datos extraídos correctamente para Australia
✅ Datos extraídos correctamente para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Bahamas no está en la lista, se omite.
⏭️ País Bahrain no está en la lista, se omite.
⏭️ País Bangladesh no está en la lista, se omite.
⏭️ País Barbados no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos extraídos correctamente para Belgium
⏭️ País Belize no está en la lista, se omite.
⏭️ País Benin no está en la lista, se omite.
⏭️ País Bolivia no está en la lista, se omite.
⏭️ País Bosnia-and-Herzegovina no está en la lista, se omite.
⏭️ País Botswana no está en la lis

In [14]:
master_df.head()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,4.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1969,NaN,NaN,NaN,NaN,NaN,76.62,NaN,NaN,NaN,NaN,...,NaN,4.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [16]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [17]:
df_paraguay.head()

Country,Paraguay
Period,
Apr 1996,4340.15
Apr 1997,5428.67
Apr 1998,6672.60
Apr 1999,6051.41
Apr 2000,6630.16


In [18]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
master_df = pd.merge(master_df, df_paraguay, how="outer", left_index=True, right_index=True)

In [19]:
master_df.head()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1969,NaN,NaN,NaN,NaN,NaN,76.62,NaN,NaN,NaN,NaN,...,4.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
unique_periods = pd.Series(master_df.index.astype(str)).unique()
print(unique_periods)

['Apr 1965' 'Apr 1966' 'Apr 1967' 'Apr 1968' 'Apr 1969' 'Apr 1970'
 'Apr 1971' 'Apr 1972' 'Apr 1973' 'Apr 1974' 'Apr 1975' 'Apr 1976'
 'Apr 1977' 'Apr 1978' 'Apr 1979' 'Apr 1980' 'Apr 1981' 'Apr 1982'
 'Apr 1983' 'Apr 1984' 'Apr 1985' 'Apr 1986' 'Apr 1987' 'Apr 1988'
 'Apr 1989' 'Apr 1990' 'Apr 1991' 'Apr 1992' 'Apr 1993' 'Apr 1994'
 'Apr 1995' 'Apr 1996' 'Apr 1997' 'Apr 1998' 'Apr 1999' 'Apr 2000'
 'Apr 2001' 'Apr 2002' 'Apr 2003' 'Apr 2004' 'Apr 2005' 'Apr 2006'
 'Apr 2007' 'Apr 2008' 'Apr 2009' 'Apr 2010' 'Apr 2011' 'Apr 2012'
 'Apr 2013' 'Apr 2014' 'Apr 2015' 'Apr 2016' 'Apr 2017' 'Apr 2018'
 'Apr 2019' 'Apr 2020' 'Apr 2021' 'Apr 2022' 'Apr 2023' 'Apr 2024'
 'Apr 2025' 'Aug 1965' 'Aug 1966' 'Aug 1967' 'Aug 1968' 'Aug 1969'
 'Aug 1970' 'Aug 1971' 'Aug 1972' 'Aug 1973' 'Aug 1974' 'Aug 1975'
 'Aug 1976' 'Aug 1977' 'Aug 1978' 'Aug 1979' 'Aug 1980' 'Aug 1981'
 'Aug 1982' 'Aug 1983' 'Aug 1984' 'Aug 1985' 'Aug 1986' 'Aug 1987'
 'Aug 1988' 'Aug 1989' 'Aug 1990' 'Aug 1991' 'Aug 1992' 'Aug 1

In [31]:
# Reset index to expose 'Period'
master_df2 = master_df.copy()
master_df2 = master_df.reset_index()

# Month to Quarter mapping
month_to_quarter = {
    "Jan": "Q1", "Feb": "Q1", "Mar": "Q1",
    "Apr": "Q2", "May": "Q2", "Jun": "Q2", "June": "Q2",
    "Jul": "Q3", "Aug": "Q3", "Sep": "Q3",
    "Oct": "Q4", "Nov": "Q4", "Dec": "Q4"
}

# Function to standardize each Period entry
def standardize_period(period):
    period = str(period).strip()

    # Match "Month Year" format
    match_month = re.match(r"^([A-Za-z]{3,4}) (\d{4})$", period)
    if match_month:
        month, year = match_month.groups()
        quarter = month_to_quarter.get(month)
        if quarter:
            return f"{quarter} {year}"

    # Match "Quarter Year" format
    match_quarter = re.match(r"^(Q[1-4]) (\d{4})$", period)
    if match_quarter:
        quarter, year = match_quarter.groups()
        return f"{quarter} {year}"

    # If no match, return None or original for inspection
    return None

In [32]:
master_df2.head()

,Period,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,Apr 1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Apr 1966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Apr 1967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Apr 1968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Apr 1969,NaN,NaN,NaN,NaN,NaN,76.62,NaN,NaN,NaN,...,4.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# Apply the function
master_df2["Standardized_Period"] = master_df2["Period"].apply(standardize_period)

In [34]:
# Drop rows where standardization failed
master_df2 = master_df2[master_df2["Standardized_Period"].notna()].copy()

In [35]:
# Split standardized period into Quarter and Year
master_df2[["Quarter", "Year"]] = master_df2["Standardized_Period"].str.extract(r"(Q[1-4]) (\d{4})")
master_df2["Year"] = master_df2["Year"].astype(int)

In [37]:
# Reorder columns
cols = ["Quarter", "Year"] + [col for col in master_df2.columns if col not in ["Period", "Month", "Quarter", "Year", "Standardized_Period"]]
master_df2 = master_df2[cols]

In [38]:
# Sort by Year and Quarter
master_df2 = master_df2.sort_values(by=["Year", "Quarter"]).reset_index(drop=True)

In [39]:
master_df2.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,Q1,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Q2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Q3,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
unique_periods = pd.Series(master_df2.index.astype(str)).unique()
print(unique_periods)

['0' '1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15'
 '16' '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29'
 '30' '31' '32' '33' '34' '35' '36' '37' '38' '39' '40' '41' '42' '43'
 '44' '45' '46' '47' '48' '49' '50' '51' '52' '53' '54' '55' '56' '57'
 '58' '59' '60' '61' '62' '63' '64' '65' '66' '67' '68' '69' '70' '71'
 '72' '73' '74' '75' '76' '77' '78' '79' '80' '81' '82' '83' '84' '85'
 '86' '87' '88' '89' '90' '91' '92' '93' '94' '95' '96' '97' '98' '99'
 '100' '101' '102' '103' '104' '105' '106' '107' '108' '109' '110' '111'
 '112' '113' '114' '115' '116' '117' '118' '119' '120' '121' '122' '123'
 '124' '125' '126' '127' '128' '129' '130' '131' '132' '133' '134' '135'
 '136' '137' '138' '139' '140' '141' '142' '143' '144' '145' '146' '147'
 '148' '149' '150' '151' '152' '153' '154' '155' '156' '157' '158' '159'
 '160' '161' '162' '163' '164' '165' '166' '167' '168' '169' '170' '171'
 '172' '173' '174' '175' '176' '177' '178' '179' '180' '181' '182

In [41]:
master_df2["Quarter"].unique().tolist()

['Q1', 'Q2', 'Q3', 'Q4']

In [42]:
master_df2[master_df2["Quarter"].isnull()]

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay


In [46]:
def convert_quarter_to_int(df, quarter_col="Quarter"):
    """
    Converts quarter strings like 'Q1' to integers (1–4) in the specified column.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        quarter_col (str): The name of the column containing quarter strings.
    
    Returns:
        pd.DataFrame: A copy of the DataFrame with the quarter column converted to integers.
    """
    quarter_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
    df = df.copy()
    df[quarter_col] = df[quarter_col].map(quarter_map)
    return df

In [48]:
master_df3 = master_df2.copy()

In [49]:
master_df3 = convert_quarter_to_int(master_df3)
master_df3.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,1,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
master_df3["Quarter"].unique().tolist()

[1, 2, 3, 4]

In [51]:
def convert_year_to_int(df, year_col="Year"):
    """
    Converts the specified year column to integers, handling strings and other types.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        year_col (str): The name of the column containing year values.
    
    Returns:
        pd.DataFrame: A copy of the DataFrame with the year column as integers.
    """
    df = df.copy()
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce").astype("Int64")
    return df

In [52]:
master_df3 = convert_year_to_int(master_df3)
master_df3.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,1,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
master_df3["Year"].unique().tolist()

[1965,
 1966,
 1967,
 1968,
 1969,
 1970,
 1971,
 1972,
 1973,
 1974,
 1975,
 1976,
 1977,
 1978,
 1979,
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024,
 2025]

In [54]:
# Guardar como archivo Parquet
master_df3.to_parquet("../../data/private_credit_values.parquet", index=False)